# 07 · Data Split — grouped and stratified 70 / 15 / 15

**In short:** the cleaned training images (PlantVillage, FieldPlant, Tanzania potato) are
divided into **train** (to learn from), **val** (to tune and calibrate) and **test** (to
report the final score). PlantDoc is never split; it stays the out-of-domain test.

**Why groups?** Photos of the same leaf or plant look almost identical. If one went to
train and its twin to test, the model would be tested on something it had already seen,
and the score would look better than it really is. So related photos are tied into a
**group**, and a whole group always goes to the same partition. Photos are tied together
when they share:

- a **PlantVillage leaf record** (4 photos per leaf; maize has no records),
- a **FieldPlant source photo** (several leaves cropped from one picture),
- a **Tanzania capture session** (photos taken minutes apart, same class),
- a **near-duplicate group** from cleaning.

**How the split is made:** the groups are dealt into 20 equal folds that each keep the same
mix of source and class (`StratifiedGroupKFold`, seed 42); 14 folds become train, 3 val, 3 test.

The notebook **stops after the Tanzania time-gap tables** so the session cut-off can be
chosen from the numbers. Runs on a CPU runtime.

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

In [ ]:
# Keep the Drive copy of the repository in sync with GitHub.
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

In [ ]:
%pip install -q -r {ML_DIR}/requirements.txt

## Load the cleaned images

Only images kept by cleaning (`keep == True`) are used.

In [ ]:
import pandas as pd

from src.data import split

manifest = pd.read_csv(config.METADATA_DIR / "clean_manifest.csv")
kept = manifest[manifest["keep"]]
pool = kept[kept["subset"] == "train_pool"].copy()
ood_test = kept[kept["subset"] == "ood_in_scope"].copy()
unseen_test = kept[kept["subset"] == "ood_unseen"].copy()

print("Training pool:", len(pool), pool["source"].value_counts().to_dict())
print("PlantDoc in scope:", len(ood_test), "| PlantDoc unseen:", len(unseen_test))

## Tanzania: look at the time gaps

A **session** is a run of photos taken close together in time; a new session starts when
the gap to the previous photo is longer than a cut-off. Sessions are found on **all 58,709
photos**, not the 3,000-photo sample: the sample keeps about 1 photo in 20, so gaps between
sampled photos look about 20 times longer than they really are.

The first table shows typical gaps (in minutes) between consecutive photos. The second shows,
for each candidate cut-off, how many sessions there are and how large the biggest one is
**within the sample**. A session holding more than a few percent of a class's sample is hard
to fit into a 15% partition.

In [ ]:
inventory = pd.read_csv(config.METADATA_DIR / "tanzania_inventory.csv", parse_dates=["capture_time"])
tz_pool = pool[pool["source"] == "tanzania"].assign(file=lambda d: d["path"].str.rsplit("/", n=1).str[-1])

print("Photos without a capture time:", inventory["capture_time"].isna().sum(), "(each becomes its own session)")
print("\nMinutes between consecutive photos:")
display(split.gap_quantiles(inventory))

CANDIDATE_CUTOFFS = [2, 5, 10, 30]
display(split.gap_table(inventory, tz_pool, CANDIDATE_CUTOFFS))

## Choose the session cut-off

Set `SESSION_GAP_MINUTES` after reading the tables above. If one class has a session too
large to balance, put a smaller cut-off for that class in `SESSION_GAP_OVERRIDES`,
e.g. `{"Potato___Early_blight": 2}`. The table printed here is the session summary for
Chapter 5.

In [ ]:
SESSION_GAP_MINUTES = None
SESSION_GAP_OVERRIDES = {}

assert SESSION_GAP_MINUTES is not None, "Choose SESSION_GAP_MINUTES from the tables above first."
gap_minutes = {c: SESSION_GAP_OVERRIDES.get(c, SESSION_GAP_MINUTES) for c in config.TANZANIA_CLASS_MAP.values()}

inventory["session"] = split.assign_sessions(inventory, gap_minutes)
tz_pool = tz_pool.merge(inventory[["class", "file", "session"]], on=["class", "file"], how="left")
assert tz_pool["session"].notna().all(), "Some sampled photos were not found in the inventory"

sessions = split.session_summary(tz_pool, gap_minutes)
display(sessions)
sessions.to_csv(config.METADATA_DIR / "tanzania_sessions_summary.csv")

## Build the groups

Each image collects its grouping keys (leaf, source photo, session, near-duplicate group);
images sharing any key, even through a chain, end up in one group.

In [ ]:
leaf_maps_dir = config.PLANTVILLAGE_DIR / "leaf_grouping" / "filtered_leafmaps"
assert leaf_maps_dir.exists(), f"Leaf records missing on Drive: {leaf_maps_dir}"
leaf_of = split.read_leaf_maps(leaf_maps_dir, config.PLANTVILLAGE_CLASSES)

file_name = pool["path"].str.rsplit("/", n=1).str[-1]
is_pv = pool["source"] == "plantvillage"
pool["leaf"] = [
    leaf_of.get((c, split.leaf_name(f))) if pv else None
    for c, f, pv in zip(pool["class"], file_name, is_pv)
]

crops = pd.read_csv(config.METADATA_DIR / "fieldplant_crops.csv")
pool["source_image"] = file_name.map(crops.set_index("crop")["source_image"]).where(pool["source"] == "fieldplant")
assert pool.loc[pool["source"] == "fieldplant", "source_image"].notna().all(), "FieldPlant crops without a source photo"

pool["session"] = pool["path"].map(tz_pool.set_index("path")["session"])
pool["group"] = split.connect(split.group_keys(pool))

print("PlantVillage images with a leaf record:", pool["leaf"].notna().sum(), "of", is_pv.sum())
display(pool[is_pv].assign(has_leaf=pool["leaf"].notna()).groupby("class")["has_leaf"].agg(["sum", "size"]))

group_sizes = pool.groupby("group").size()
print("Groups:", len(group_sizes), "| median size:", int(group_sizes.median()), "| largest:", group_sizes.max())
display(pool.groupby("source")["group"].nunique().rename("groups"))

## Split

In [ ]:
ratios = (config.TRAIN_RATIO, config.VAL_RATIO, config.TEST_RATIO)
pool["partition"] = split.assign_partitions(pool, config.SPLIT_FOLDS, ratios, config.RANDOM_SEED)

display(pool["partition"].value_counts().reindex(split.PARTITIONS).rename("images"))
display((split.shares(pool, "class") * 100).round(1))
display((split.shares(pool, "source") * 100).round(1))

## Checks

Each check stops the notebook with an error if it fails.

In [ ]:
for line in split.check(pool, set(ood_test["path"]) | set(unseen_test["path"]), ratios, config.SPLIT_TOLERANCE):
    print("PASS:", line)

## Save the split

`train.csv`, `val.csv`, `test.csv`, `ood_test.csv` (PlantDoc, in-scope classes) and
`unseen_test.csv` (PlantDoc, unseen tomato diseases) go to `datasets/splits/`, with paths
relative to the raw data folder. `split_summary.csv` has images per source, class and
partition, for per-source reporting in Chapter 5.

In [ ]:
config.SPLITS_DIR.mkdir(parents=True, exist_ok=True)
columns = ["path", "class", "source", "group"]

for name in split.PARTITIONS:
    pool.loc[pool["partition"] == name, columns].to_csv(config.SPLITS_DIR / f"{name}.csv", index=False)
ood_test.assign(group=pd.NA)[columns].to_csv(config.SPLITS_DIR / "ood_test.csv", index=False)
unseen_test.assign(group=pd.NA)[columns].to_csv(config.SPLITS_DIR / "unseen_test.csv", index=False)

summary = split.summary_table(pool)
summary.to_csv(config.SPLITS_DIR / "split_summary.csv", index=False)
display(summary)
print("Saved to", config.SPLITS_DIR, ":", sorted(p.name for p in config.SPLITS_DIR.iterdir()))

## Make sure Drive has saved everything

Colab uploads files to Drive in the background; this waits until the upload has finished
(see DEF-01), then reconnects Drive for the push.

In [ ]:
drive.flush_and_unmount()
drive.mount("/content/drive")
print("Drive synced and remounted.")

## Save to GitHub

Commits the changed notebook to a new branch and prints a link to open the pull
request (`main` is protected). Change the branch name if this cell has already been
run once. Save the notebook (Ctrl+S) before running this cell.

In [ ]:
from google.colab import userdata

from src import github_sync

print(github_sync.status(REPO_DIR))
print()
print(github_sync.push(
    REPO_DIR,
    branch="colab/data-split",
    message="Split the training pool into grouped, stratified train, val and test",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
))